# Ethiopia — Task 2: Profiling, cleaning, and EDA



## Setup: imports and repo root

We import `find_data_start_row` and `ZSCORE_COLUMNS` from the project library only to handle the NASA CSV preamble and to list the seven variables for Z-scores; **all Task 2 logic is written explicitly** in the cells below.

**Environment:** This cell needs `pandas` and `numpy` in the **active Jupyter kernel**.

- If you see **`No module named 'numpy'`** (or `pandas`), the kernel is **not** your project venv. In Cursor: **Notebook: Select Notebook Kernel** → **Python (climate-week0 venv)** or **`venv\Scripts\python.exe`**. Then run **Python: Select Interpreter** to the same path so terminals and notebooks match.
- Install deps once from the repo root: `python -m pip install -r requirements.txt` (with that venv activated or using `venv\Scripts\python.exe -m pip ...`).
- Section 8 plots import `matplotlib` in their cell (same kernel must have `requirements.txt` installed).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _repo_root() -> Path:
    """Folder that contains `src/climate_cleaning.py` (works if cwd is repo root, `notebooks/`, or elsewhere inside the repo)."""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "src" / "climate_cleaning.py").is_file():
            return d
    raise FileNotFoundError(
        "Could not find project root (expected src/climate_cleaning.py). "
        "Open the climate-challenge-week0 folder in Cursor or cd there before running."
    )


ROOT = _repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.climate_cleaning import ZSCORE_COLUMNS, find_data_start_row

## 1. Data loading and date parsing

- Load with **`pd.read_csv`**. NASA POWER exports include lines before the `YEAR,DOY,...` header — use **`skiprows=find_data_start_row(path)`**. If your file has no preamble, use `skiprows=0`.
- Add **`Country`**.
- **`date`:** `pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")` (Julian day of year).
- **`Month`** for seasonal analysis.

In [9]:
COUNTRY = "Ethiopia"
RAW_PATH = ROOT / "data" / "ethiopia.csv"
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Expected raw CSV at {RAW_PATH}. Add data/ethiopia.csv (see README)."
    )

skiprows = find_data_start_row(RAW_PATH)
df = pd.read_csv(RAW_PATH, skiprows=skiprows)
df["Country"] = COUNTRY
df["date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["Month"] = df["date"].dt.month
df.head()

NameError: name 'ROOT' is not defined

## 2. Sentinel values (before any statistics)

NASA POWER uses **`-999`** for missing or out-of-range values. Replace with **`np.nan` across the entire DataFrame** before `describe()`, missingness rates, or Z-scores.

In [ ]:
df = df.replace(-999, np.nan)

## 3. Duplicate rows

Run **`df.duplicated().sum()`** before dropping. A **full-row duplicate** means every column matches another row (including `Country`, `YEAR`, `DOY`, and all weather fields).

We also check duplicates on **`["YEAR", "DOY"]`** — non-zero counts mean the same calendar day appears more than once (worth investigating if the file was appended twice).

In [ ]:
dup_full = int(df.duplicated().sum())
dup_key = int(df.duplicated(subset=["YEAR", "DOY"]).sum())
print(f"Full-row duplicates (to drop with keep='first'): {dup_full}")
print(f"Duplicates on [YEAR, DOY] only: {dup_key}")

df = df.drop_duplicates().reset_index(drop=True)

### Duplicate documentation (this run on `data/ethiopia.csv`)

- **Full-row duplicates (`dup_full`):** 0 — no rows were identical across every column.
- **Duplicates on `[YEAR, DOY]` (`dup_key`):** 0 — no repeated calendar days in this extract.
- **Which columns are involved when dups exist?** For full-row duplicates, *all* columns match between the repeated rows (whole-record repetition), not a single “duplicate column.”
- **If `dup_key > 0` but `dup_full == 0`:** inspect with `df[df.duplicated(subset=['YEAR','DOY'], keep=False)].sort_values(['YEAR','DOY'])` — same date, differing values elsewhere.

## 4. Summary statistics (`describe`)

Numeric columns only. Interpret the table in the markdown cell below.

In [ ]:
numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
df[numeric_cols].describe().T

### Interpretation (`describe`)

Use your table above for Ethiopia’s grid point:

- **Temperatures (`T2M`, `T2M_MAX`, `T2M_MIN`):** Central values and spread reflect local climate and seasonality; diurnal range is reflected in max–min separation.
- **`PRECTOTCORR`:** The mean is often **small** compared to wet-day totals because many days are **dry (near zero)** — rainfall is **right-skewed**, not Gaussian.
- **`RH2M`:** Humidity co-varies with temperature and rain; interpret joint behaviour rather than one number in isolation.
- **Winds (`WS2M`, `WS2M_MAX`):** Gust max should generally exceed daily mean wind.
- **`PS` (surface pressure):** Depends on site elevation; compare across countries after repeating this notebook.

**COP32 / trends framing:** Means alone hide extremes. Variability, tails (heat, drought, intense rain), and changes in seasonality matter for adaptation and loss-and-damage discussions — the sections below and the optional plots support that narrative.

## 5. Missing values

**`df.isna().sum()`** and **percentage per column**. Flag columns with **> 5%** missing.

In [ ]:
miss_n = df.isna().sum()
miss_pct = 100 * df.isna().mean()
miss_tbl = (
    pd.DataFrame({"n_missing": miss_n, "pct_missing": miss_pct})
    .sort_values("pct_missing", ascending=False)
)
display(miss_tbl)
over5 = miss_tbl[miss_tbl["pct_missing"] > 5]
print("Columns with >5% missing:")
display(over5)

### Missing data — what >5% means

High missing rates can indicate **retrieval gaps**, **variables undefined** for some periods, or **persistent sentinel replacement** for that field. Aggregates (annual means, trends) can be **biased** if missingness is not random; prefer reporting uncertainty or limiting analysis to well-observed variables and periods.

## 6. Outlier detection (Z-scores)

For each variable in **`T2M`, `T2M_MAX`, `T2M_MIN`, `PRECTOTCORR`, `RH2M`, `WS2M`, `WS2M_MAX`** that exists in the file: compute **Z = (x − μ) / σ** with **population** σ (`ddof=0`), ignoring `NaN`. Flag **|Z| > 3** and report counts.

In [ ]:
z_cols = [c for c in ZSCORE_COLUMNS if c in df.columns]
any_outlier = np.zeros(len(df), dtype=bool)

for col in z_cols:
    s = pd.to_numeric(df[col], errors="coerce")
    mu = s.mean()
    sigma = s.std(ddof=0)
    if sigma == 0 or np.isnan(sigma):
        df[f"z_outlier_{col}"] = False
        print(f"{col}: |Z|>3 count = 0 (undefined or zero std)")
        continue
    z = (s - mu) / sigma
    flag = z.abs() > 3
    df[f"z_outlier_{col}"] = flag.fillna(False)
    n = int(flag.fillna(False).sum())
    print(f"{col}: |Z|>3 count = {n}")
    any_outlier |= df[f"z_outlier_{col}"].to_numpy()

print(f"Rows with |Z|>3 in at least one variable: {int(any_outlier.sum())}")

## 7. Cleaning decision: outliers

**Decision: retain** flagged rows in this analysis.

**Reasoning:** For climate observations, large |Z| often corresponds to **real extremes** (heat waves, very dry days, heavy rain) that are central to **COP32-relevant** risk and adaptation discourse — not necessarily data errors. Dropping them would **understate variability** and bias trend and extreme-value narratives. If a row is **physically impossible** (e.g. inconsistent triples after QA), treat that as **data error**, not a Gaussian outlier.

We keep the boolean **`z_outlier_*`** columns for optional sensitivity analysis (e.g. compare plots with and without flagged days).

## 8. Optional EDA: seasonality and COP32 narrative hooks

Quick visuals using **`date`** / **`Month`** after profiling. Adjust variable names if your CSV differs.

Requires **`matplotlib`** (included in `requirements.txt`). The next cell imports it; if that fails, install deps from the repo root and switch the notebook kernel to that environment.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)

plot_df = df.set_index("date").sort_index()
if "T2M" in plot_df.columns:
    monthly_t = plot_df["T2M"].resample("ME").mean()
    fig, ax = plt.subplots()
    monthly_t.plot(ax=ax, color="firebrick")
    ax.set_title("Ethiopia — monthly mean T2M (°C)")
    ax.set_ylabel("T2M")
    plt.tight_layout()
    plt.show()

if "PRECTOTCORR" in plot_df.columns:
    monthly_p = plot_df["PRECTOTCORR"].resample("ME").sum()
    fig, ax = plt.subplots()
    monthly_p.plot(ax=ax, color="steelblue", drawstyle="steps-post")
    ax.set_title("Ethiopia — monthly total PRECTOTCORR (mm)")
    plt.tight_layout()
    plt.show()

if "T2M" in df.columns and "Month" in df.columns:
    fig, ax = plt.subplots()
    df.boxplot(column="T2M", by="Month", ax=ax)
    plt.suptitle("")
    ax.set_title("Ethiopia — T2M by calendar month")
    plt.tight_layout()
    plt.show()

### Notes for your report (COP32 / African climate narrative)

Use your plots together with the profiling above. For **this run** (after sentinel cleanup): **132** distinct days had **|Z|>3** in at least one of the seven core variables — dominated by **PRECTOTCORR** (95 days), then **T2M_MIN** (18), **RH2M** (13), etc. Treat these as **potential extremes** (heavy rain or dry/cool tails), not automatic errors — consistent with the “retain outliers” decision.

**Seasonality:** The monthly **T2M** line and **T2M-by-Month** boxplots show how the thermal season cycle aligns with insolation and the regional rainy seasons; **PRECTOTCORR** monthly totals highlight wet vs dry months — relevant for **agriculture, water, and hydropower** stressors often discussed in UNFCCC adaptation finance.

**Interannual variability:** Long runs of low monthly rain or warm anomalies in the time series can frame **loss and damage** and **early warning** needs without over-interpreting a single grid point as a national average.

**For COP:** Connect clearer **variability and extremes** (not only means) to **resilience investment**, **climate finance**, and **ambition** in the lead-up to COP32; cite your figures and the Z-score counts as evidence of tail behaviour in the daily record.